# Owner pipeline (EDGAR companies)

Builds the EDGAR company reference tables from the SEC EDGAR bulk submissions data: one JSON file per filer
→ `input_1.edgar_submissions` (one row per CIK, as reported by EDGAR) + `input_1.owner_holdings` (coverage in the
holdings table) → `clean_up_1.edgar_company` (one row per company: CIK, name, type, SIC, EIN, LEI, tickers) and
`clean_up_1.edgar_company_name` (current and former names). The company table holds both sides of a holding:
the **owners** (13F filers, joined to the holdings by `cik`) and the **issuers** (filers with a SIC code or ticker,
used by `5_industry_pipeline.ipynb`). Every block shows the database feedback (status message, affected rows,
duration, result set) and is written to the run log `logs/owner/<RUN_ID>/`, so runs can be compared later
(section 6). Execution and logging code lives in `src/pipeline_helpers.py`.

Source: `submissions.zip` from https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip, extracted
to `Thesis/submission/submissions`. Replaces `clean_up_1.owner` / `clean_up_1.owner_former_name` (earlier version
of this pipeline) and `src/database_ownertable/create_ownertable.ipynb`.

| Section | Phase label | What happens |
|---|---|---|
| 0.2 Database connection | `connection` | read-only check, then working connection |
| 1. Submissions table | `owner_create` | input table for the parsed submissions |
| 2. Load submissions | `owner_load` | scan the JSON files: 13F filers, filers with SIC / ticker, every CIK of the holdings |
| 3. Submissions profile | `owner_profile` | totals, entity types, EIN / LEI quality, 13F history |
| 4. Build company tables | `owner_build` | holdings coverage per CIK, one row per company, current and former names |
| 5. Checks | `owner_checks` | coverage of the holdings, report windows, renamed owners |

> **The cells modify the database.** `input_1.owner_holdings`, `clean_up_1.edgar_company` and
> `clean_up_1.edgar_company_name` are dropped and rebuilt in section 4; 4.4 drops the replaced owner tables.

## 0. Setup

### 0.1 Configuration

* `SUBMISSIONS_DIR`: the extracted `submissions.zip`
* `SCAN_WORKERS`: processes for the file scan (`None` = one per CPU)

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent                       # .../Database/Initialisation (notebooks run from notebooks/)
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "config")]

from pipeline_helpers import (EdgarSubmissionLoader, PipelineRun, compare_runs, expected_mismatches,
                              load_logs, phase_durations, steps_table)
from local_config import DSN, SUBMISSIONS_DIR   # machine-specific, git-ignored (template: local_config.example.py)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

LOG_ROOT = ROOT / "logs" / "owner"

SUBMISSIONS_TABLE    = "input_1.edgar_submissions"
COMPANY_TABLE        = "clean_up_1.edgar_company"
NAME_TABLE           = "clean_up_1.edgar_company_name"
OWNER_HOLDINGS_TABLE = "input_1.owner_holdings"
HOLDINGS_TABLE       = "clean_up_1.holdings_clean"
FILINGS_TABLE        = "clean_up_1.filings"

# SUBMISSIONS_DIR: extracted submissions.zip (SEC EDGAR bulk data)

SCAN_WORKERS = None   # None = one process per CPU

# The company tables are small (~90k rows); exact counts are cheap
p = PipelineRun(LOG_ROOT, dsn=DSN, max_rows=1000, exact_counts=True)
print("RUN_ID :", p.run_id)
print("LOG_DIR:", p.log_dir)

RUN_ID : 20261029_155949
LOG_DIR: <THESIS_DIR>/src/Clean_solution/Database/Initialisation/logs/owner/20261029_155949


### 0.2 Database connection

First a read-only check on a separate connection (`READ ONLY` transaction, rolled back): server, database,
user, schema privileges, current state of the tables. The holdings and filings tables must exist (built by
`1_holdings_pipeline.ipynb`). The working connection (autocommit) is opened only if the check passes.

In [2]:
CONNECTION_OK = p.check_connection(DSN, schemas=["input_1", "clean_up_1"],
                                   tables=[SUBMISSIONS_TABLE, OWNER_HOLDINGS_TABLE, COMPANY_TABLE, NAME_TABLE,
                                           HOLDINGS_TABLE, FILINGS_TABLE])

✅ **[connection] connection_check_server**: `ok` | 0.1s

> READ ONLY transaction, rolled back. Nothing committed.

,server_version,database,db_user,is_standby,txn_read_only,db_size,other_sessions
0,PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04....,Holdings_v2,postgres,False,on,70 GB,0


✅ **[connection] connection_check_schemas**: `ok`

,schema_name,schema_exists,can_create,can_use
0,input_1,True,True,True
1,clean_up_1,True,True,True


✅ **[connection] connection_check_tables**: `ok`

> Existing tables will be dropped and recreated by the pipeline.

,table_name,exists,unlogged,est_rows,total_size
0,clean_up_1.edgar_company,True,False,88009,19 MB
1,clean_up_1.edgar_company_name,True,False,118248,14 MB
2,input_1.edgar_submissions,True,False,88009,25 MB
3,input_1.owner_holdings,True,False,10661,832 kB
4,clean_up_1.holdings_clean,True,False,97681472,25 GB
5,clean_up_1.filings,True,False,301764,52 MB


In [3]:
assert CONNECTION_OK, "Connection check failed, see connection_check_server above"
p.connect()

Connected: user=postgres password=xxx dbname=Holdings_v2 host=localhost port=5432


## 1. Submissions table

`input_1.edgar_submissions` holds one row per CIK as EDGAR reports it today: the filer's **current** name,
entity type, SIC, EIN, LEI, tickers and exchanges, state and business address, the former names (JSON), and the
13F history counted from all its filings:

* `n_13f_hr`, `n_13f_hr_a`, `n_13f_nt`, `n_13f_nt_a`: number of filings per 13F form
* `first_13f_filed`, `last_13f_filed`: filing dates of the first / last 13F form of any type
* `first_13f_report`, `last_13f_report`: report periods of the first / last 13F-HR or 13F-HR/A
  (notices have no holdings)
* `filings_total`: all EDGAR filings of the CIK, `overflow_files`: older-filing files read for it
* `forced`: the CIK was loaded because it appears in the holdings filings (see 2)

Values are stored as EDGAR delivers them (empty strings become `NULL`); typing and cleaning happen in section 4.

In [4]:
sub = p.stage("owner_create")

sub.run("drop_submissions_table", f"DROP TABLE IF EXISTS {SUBMISSIONS_TABLE};")

sub.run("create_submissions_table", f"""
    CREATE TABLE {SUBMISSIONS_TABLE} (
        cik                        INTEGER  PRIMARY KEY,
        name                       TEXT,
        entity_type                TEXT,
        sic                        TEXT,
        sic_description            TEXT,
        ein                        TEXT,
        lei                        TEXT,
        category                   TEXT,
        tickers                    TEXT[]   NOT NULL,
        exchanges                  TEXT[]   NOT NULL,
        state_of_incorporation     TEXT,
        fiscal_year_end            TEXT,
        business_city              TEXT,
        business_state             TEXT,
        business_state_description TEXT,
        former_names               JSONB    NOT NULL,
        n_13f_hr                   INTEGER  NOT NULL,
        n_13f_hr_a                 INTEGER  NOT NULL,
        n_13f_nt                   INTEGER  NOT NULL,
        n_13f_nt_a                 INTEGER  NOT NULL,
        first_13f_filed            DATE,
        last_13f_filed             DATE,
        first_13f_report           DATE,
        last_13f_report            DATE,
        filings_total              INTEGER  NOT NULL,
        overflow_files             SMALLINT NOT NULL,
        overflow_missing           SMALLINT NOT NULL,
        forced                     BOOLEAN  NOT NULL,
        source_file                TEXT     NOT NULL
    );
""")

✅ **[owner_create] drop_submissions_table**: `ok` | `DROP TABLE` | 0.0s

✅ **[owner_create] create_submissions_table**: `ok` | `CREATE TABLE` | 0.0s

""


## 2. Load the submissions

Every filer has a main file `CIK##########.json` with its latest ~1,000 filings; older filings are in overflow
files `CIK##########-submissions-NNN.json` (large filers such as Vanguard have a dozen). A CIK is kept when

* **any** of its files contains a 13F form (an owner). The forms are counted from the parsed `form` arrays, so a
  "13F-HR" somewhere in a description does not count (the old notebook searched the raw text), or
* it has a **SIC code** (`0000` counts as none) or a **ticker** (an issuer, for the industry classification).

The CIKs of `clean_up_1.filings` are passed as `extra_ciks`: they are loaded in any case, so every owner of the
holdings gets a row.

The scan (~975k files, ~7.5 GB) runs in parallel and takes about 1–2 minutes. The table is replaced only after
the scan finished: interrupting the kernel (■) during the scan leaves the previous load intact.

In [5]:
load = p.stage("owner_load")

holdings_ciks = load.run("holdings_ciks", f"SELECT DISTINCT cik FROM {FILINGS_TABLE} ORDER BY cik;",
                         fetch_all=True, expected=10_713,
                         note="every CIK with a 13F filing in the holdings pipeline (incl. superseded filings)")
EXTRA_CIKS = holdings_ciks["cik"].astype(int).tolist()

loader = EdgarSubmissionLoader(load, SUBMISSIONS_DIR, SUBMISSIONS_TABLE, workers=SCAN_WORKERS)
loader.preflight(EXTRA_CIKS)

✅ **[owner_load] holdings_ciks**: `ok` | `SELECT 10710` | rows: **10,710** | expected: 10,713 (diff -3) | 0.3s

> every CIK with a 13F filing in the holdings pipeline (incl. superseded filings)

,cik
0,2230
1,3133
2,3520
3,3521
4,4962
...,...
10705,1846943
10706,1846991
10707,1846994
10708,1847258


✅ **[owner_load] preflight_submissions**: `ok`

> <THESIS_DIR>/submission/submissions  |  extra CIKs without file (first 20): none

,check,n
0,main files,974989
1,overflow files,5302
2,CIKs with overflow files,4128
3,overflow without main file,0
4,unexpected file names,1
5,extra CIKs,10710
6,extra CIKs without main file,0


True

In [6]:
submissions = loader.run(EXTRA_CIKS)   # previous test: 88,009 CIKs (20,108 13F filers, 68,477 with SIC or ticker, 576 both)

<THESIS_DIR>/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
submissions: 100%|██████████| 974989/974989 [01:11<00:00, 13541.75file/s]


✅ **[owner_load] load_submissions**: `ok` | rows: **88,009** | 75.3s

> 974,989 main files scanned | 10,710 extra CIKs

,group,ciks
0,13F filers,20108
1,with 13F-HR or 13F-HR/A,16085
2,notices only (13F-NT),4023
3,filers with SIC or ticker,68477
4,of them 13F filers,576
5,extra CIKs without 13F / SIC / ticker,0
6,parse errors,0


## 3. Submissions profile

In [7]:
prof = p.stage("owner_profile")

prof.row_estimate(SUBMISSIONS_TABLE, "submissions")
prof.run("submissions_totals", f"""
    SELECT count(*)                                                         AS ciks,
           count(*) FILTER (WHERE n_13f_hr + n_13f_hr_a > 0)                AS holdings_reporters,
           count(*) FILTER (WHERE n_13f_hr + n_13f_hr_a = 0
                              AND n_13f_nt + n_13f_nt_a > 0)                AS notices_only,
           count(*) FILTER (WHERE nullif(sic, '0000') IS NOT NULL)          AS with_sic,
           count(*) FILTER (WHERE cardinality(tickers) > 0)                 AS with_ticker,
           count(*) FILTER (WHERE forced)                                   AS in_holdings_filings,
           count(*) FILTER (WHERE forced AND n_13f_hr + n_13f_hr_a = 0)     AS in_holdings_no_13f_hr,
           sum(n_13f_hr)                                                    AS f13_hr,
           sum(n_13f_hr_a)                                                  AS f13_hr_a,
           sum(n_13f_nt)                                                    AS f13_nt,
           sum(n_13f_nt_a)                                                  AS f13_nt_a,
           count(*) FILTER (WHERE overflow_files > 0)                       AS with_overflow,
           count(*) FILTER (WHERE overflow_missing > 0)                     AS overflow_missing
    FROM {SUBMISSIONS_TABLE};
""", note="Previous test: 88,009 CIKs | 16,085 holdings reporters | 4,023 notices only | with_sic 67,406 | with_ticker 8,160 | 10,713 in holdings filings, all with a 13F-HR | overflow_missing 0")

✅ **[owner_profile] submissions_count_exact**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,n
0,88009


✅ **[owner_profile] submissions_totals**: `ok` | `SELECT 1` | rows: **1** | 0.0s

> Previous test: 88,009 CIKs | 16,085 holdings reporters | 4,023 notices only | with_sic 67,406 | with_ticker 8,160 | 10,713 in holdings filings, all with a 13F-HR | overflow_missing 0

,ciks,holdings_reporters,notices_only,with_sic,with_ticker,in_holdings_filings,in_holdings_no_13f_hr,f13_hr,f13_hr_a,f13_nt,f13_nt_a,with_overflow,overflow_missing
0,88009,16085,4023,67406,8160,10710,0,444117,32524,130965,2151,3533,0


,ciks,holdings_reporters,notices_only,with_sic,with_ticker,in_holdings_filings,in_holdings_no_13f_hr,f13_hr,f13_hr_a,f13_nt,f13_nt_a,with_overflow,overflow_missing
0,88009,16085,4023,67406,8160,10710,0,444117,32524,130965,2151,3533,0


### Entity type, EIN and LEI

EDGAR's `entityType` is coarse: almost all 13F filers are `other`, most issuers `operating`. The EIN is the
9-digit IRS number: `000000000` is a placeholder (foreign filers, filers that never gave one), 8-digit values lost
their leading zero. The LEI is only present for filers that reported it in recent filings.

In [8]:
prof.run("entity_types", f"""
    SELECT coalesce(entity_type, '(null)')                         AS entity_type,
           count(*)                                                AS ciks,
           count(*) FILTER (WHERE n_13f_hr + n_13f_hr_a > 0)       AS holdings_reporters,
           count(*) FILTER (WHERE nullif(sic, '0000') IS NOT NULL) AS with_sic,
           count(*) FILTER (WHERE cardinality(tickers) > 0)        AS with_ticker,
           count(*) FILTER (WHERE forced)                          AS in_holdings_filings
    FROM {SUBMISSIONS_TABLE}
    GROUP BY 1
    ORDER BY 2 DESC;
""")

✅ **[owner_profile] entity_types**: `ok` | `SELECT 3` | rows: **3** | 0.0s

,entity_type,ciks,holdings_reporters,with_sic,with_ticker,in_holdings_filings
0,operating,44191,351,44126,5530,289
1,other,43664,15662,23254,2619,10360
2,investment,154,72,26,11,61


,entity_type,ciks,holdings_reporters,with_sic,with_ticker,in_holdings_filings
0,operating,44191,351,44126,5530,289
1,other,43664,15662,23254,2619,10360
2,investment,154,72,26,11,61


In [9]:
prof.run("identifier_quality", f"""
    SELECT count(*) FILTER (WHERE ein IS NULL)                              AS ein_null,
           count(*) FILTER (WHERE ein ~ '^0+$')                             AS ein_zeros,
           count(*) FILTER (WHERE ein ~ '^[0-9]{{9}}$' AND ein !~ '^0+$')   AS ein_valid,
           count(*) FILTER (WHERE ein ~ '^[0-9]{{8}}$')                     AS ein_8_digits,
           count(*) FILTER (WHERE ein !~ '^[0-9]{{8,9}}$')                  AS ein_other,
           count(*) FILTER (WHERE lei IS NOT NULL)                          AS lei_present,
           count(*) FILTER (WHERE upper(lei) !~ '^[A-Z0-9]{{20}}$')         AS lei_invalid,
           count(*) FILTER (WHERE name IS NULL)                             AS name_null,
           count(*) FILTER (WHERE sic IS NOT NULL)                          AS sic_present
    FROM {SUBMISSIONS_TABLE};
""")

✅ **[owner_profile] identifier_quality**: `ok` | `SELECT 1` | rows: **1** | 0.1s

,ein_null,ein_zeros,ein_valid,ein_8_digits,ein_other,lei_present,lei_invalid,name_null,sic_present
0,13956,17913,55963,175,13,258,13,1,67463


,ein_null,ein_zeros,ein_valid,ein_8_digits,ein_other,lei_present,lei_invalid,name_null,sic_present
0,13956,17913,55963,175,13,258,13,1,67463


In [10]:
prof.run("first_13f_per_year", f"""
    -- year of the first 13F-HR report period: when owners enter the 13F universe
    SELECT extract(year FROM first_13f_report)::int AS year,
           count(*)                                 AS new_reporters,
           count(*) FILTER (WHERE forced)           AS in_holdings_filings
    FROM {SUBMISSIONS_TABLE}
    WHERE first_13f_report IS NOT NULL
    GROUP BY 1
    ORDER BY 1;
""")

✅ **[owner_profile] first_13f_per_year**: `ok` | `SELECT 37` | rows: **37** | 0.0s

,year,new_reporters,in_holdings_filings
0,1987,1,1
1,1990,1,1
2,1991,1,1
3,1992,2,2
4,1994,1,1
5,1995,1,1
6,1996,2,2
7,1997,13,13
8,1998,107,102
9,1999,2051,2002


,year,new_reporters,in_holdings_filings
0,1987,1,1
1,1990,1,1
2,1991,1,1
3,1992,2,2
4,1994,1,1
5,1995,1,1
6,1996,2,2
7,1997,13,13
8,1998,107,102
9,1999,2051,2002


## 4. Build the company tables

### 4.1 Holdings coverage

`input_1.owner_holdings`: one row per CIK of `clean_up_1.holdings_clean` (after all cleaning steps of the holdings
pipeline) with its first / last report date, number of quarters and holding rows. Used to build the company table
and by the checks in section 5. The aggregate reads the whole holdings table (~125M rows); it is grouped by
(`cik`, `rdate`) first so the distinct quarters are counted on ~300k groups instead of sorting every row.

In [11]:
build = p.stage("owner_build")

In [12]:
build.run("create_owner_holdings", f"""
    DROP TABLE IF EXISTS {OWNER_HOLDINGS_TABLE};

    CREATE TABLE {OWNER_HOLDINGS_TABLE} (
        cik          INTEGER  PRIMARY KEY,
        first_rdate  DATE     NOT NULL,
        last_rdate   DATE     NOT NULL,
        quarters     SMALLINT NOT NULL,
        rows         BIGINT   NOT NULL
    );
""")

build.run("insert_owner_holdings", f"""
    INSERT INTO {OWNER_HOLDINGS_TABLE} (cik, first_rdate, last_rdate, quarters, rows)
    WITH q AS (
        -- one row per owner and report date first: cheap distinct count of quarters
        SELECT cik, rdate, count(*) AS n
        FROM {HOLDINGS_TABLE}
        GROUP BY cik, rdate
    )
    SELECT cik, min(rdate), max(rdate), count(*), sum(n)
    FROM q
    GROUP BY cik;
""", expected=10_661)

✅ **[owner_build] create_owner_holdings**: `ok` | `CREATE TABLE` | 0.0s

✅ **[owner_build] insert_owner_holdings**: `ok` | `INSERT 0 10681` | rows: **10,681** | expected: 10,661 (diff +20) | 100.9s

""


### 4.2 Company table

`clean_up_1.edgar_company` has **one row per company** with its identity only: every CIK of the submissions table
and every CIK of the holdings (full join, so an owner missing from EDGAR keeps its holdings with `name` `NULL`).

* `name`: the **current** EDGAR name; all names with their periods are in `clean_up_1.edgar_company_name` (4.3)
* `entity_type`: lower case (`other`, `operating`, `investment`)
* `sic`: `NULL` for the placeholder `0000`
* `ein`: 9 digits; 8-digit values padded with a leading zero, placeholders (`000000000`) → `NULL`
* `lei`: upper case, `NULL` unless 20 alphanumeric characters
* `tickers`, `exchanges`: today's tickers and their exchanges (same order), empty if not listed
* `is_owner`: the CIK has holdings in `clean_up_1.holdings_clean` (a row in `input_1.owner_holdings`)

Address, state, 13F filing counts and report periods stay in `input_1.edgar_submissions`; holdings coverage in
`input_1.owner_holdings`.

In [13]:
build.run("create_company_table", f"""
    DROP TABLE IF EXISTS {NAME_TABLE};
    DROP TABLE IF EXISTS {COMPANY_TABLE};

    CREATE TABLE {COMPANY_TABLE} (
        cik              INTEGER     PRIMARY KEY,
        name             TEXT,
        entity_type      TEXT,
        sic              VARCHAR(4),
        sic_description  TEXT,
        ein              CHAR(9)     CHECK (ein ~ '^[0-9]{{9}}$'),
        lei              CHAR(20)    CHECK (lei ~ '^[A-Z0-9]{{20}}$'),
        tickers          TEXT[]      NOT NULL DEFAULT '{{}}',
        exchanges        TEXT[]      NOT NULL DEFAULT '{{}}',
        is_owner         BOOLEAN     NOT NULL,
        is_13f_filler    BOOLEAN     NOT NULL
    );
""")

build.run("insert_company", f"""
    INSERT INTO {COMPANY_TABLE} (cik, name, entity_type, sic, sic_description, ein, lei, tickers, exchanges,
                                 is_owner, is_13f_filler)
    SELECT coalesce(s.cik, h.cik),
           s.name,
           lower(s.entity_type),
           nullif(s.sic, '0000'),
           CASE WHEN nullif(s.sic, '0000') IS NOT NULL THEN s.sic_description END,
           CASE WHEN s.ein ~ '^[0-9]{{8,9}}$' AND s.ein !~ '^0+$' THEN lpad(s.ein, 9, '0') END,
           CASE WHEN upper(s.lei) ~ '^[A-Z0-9]{{20}}$' THEN upper(s.lei) END,
           coalesce(s.tickers, '{{}}'),
           coalesce(s.exchanges, '{{}}'),
           h.cik IS NOT NULL,
           s.n_13f_hr + s.n_13f_hr_a + s.n_13f_nt + s.n_13f_nt_a > 0 
    FROM {SUBMISSIONS_TABLE} s
    FULL JOIN {OWNER_HOLDINGS_TABLE} h ON h.cik = s.cik;
""", expected=88_009, note="every holdings CIK is in the submissions, so rows = submissions rows")

✅ **[owner_build] create_company_table**: `ok` | `CREATE TABLE` | 0.0s

✅ **[owner_build] insert_company**: `ok` | `INSERT 0 88009` | rows: **88,009** | expected: 88,009 (diff +0) | 0.3s

> every holdings CIK is in the submissions, so rows = submissions rows

""


### 4.3 Company names

`clean_up_1.edgar_company_name`: one row per name a CIK used, with the period:

* the **current** name (`is_current`), valid since the end of the last former name (`name_to` `NULL`)
* every **former** name from EDGAR (`from` / `to`); the list can contain the current name as well

Used to find the name an owner reported under (renamed owners, section 5) and by the issuer matching of the
industry pipeline.

In [14]:
build.run("create_name_table", f"""
    CREATE TABLE {NAME_TABLE} (
        id          BIGSERIAL PRIMARY KEY,
        cik         INTEGER   NOT NULL REFERENCES {COMPANY_TABLE} (cik),
        name        TEXT      NOT NULL,
        name_from   DATE,
        name_to     DATE,
        is_current  BOOLEAN   NOT NULL
    );
""")

build.run("insert_names", f"""
    INSERT INTO {NAME_TABLE} (cik, name, name_from, name_to, is_current)
    WITH f AS (
        SELECT s.cik,
               trim(f.name)               AS name,
               left(f."from", 10)::date   AS name_from,     -- '1999-04-12T04:00:00.000Z' -> 1999-04-12
               left(f."to",   10)::date   AS name_to
        FROM {SUBMISSIONS_TABLE} s
        CROSS JOIN LATERAL jsonb_to_recordset(s.former_names) AS f(name text, "from" text, "to" text)
        WHERE nullif(trim(f.name), '') IS NOT NULL
    )
    SELECT s.cik, trim(s.name), l.last_to, NULL, true
    FROM {SUBMISSIONS_TABLE} s
    LEFT JOIN (SELECT cik, max(name_to) AS last_to FROM f GROUP BY cik) l USING (cik)
    WHERE nullif(trim(s.name), '') IS NOT NULL
    UNION ALL
    SELECT cik, name, name_from, name_to, false
    FROM f
    ORDER BY 1, 5, 3;
""", expected=118_248)

✅ **[owner_build] create_name_table**: `ok` | `CREATE TABLE` | 0.0s

✅ **[owner_build] insert_names**: `ok` | `INSERT 0 118248` | rows: **118,248** | expected: 118,248 (diff +0) | 0.8s

""


In [15]:
build.run("idx_company_name", f"CREATE INDEX IF NOT EXISTS edgar_company_name_idx ON {COMPANY_TABLE} (name);")
build.run("idx_company_sic", f"CREATE INDEX IF NOT EXISTS edgar_company_sic_idx ON {COMPANY_TABLE} (sic);")
build.run("idx_name_cik", f"CREATE INDEX IF NOT EXISTS edgar_company_name_cik_idx ON {NAME_TABLE} (cik);")
build.run("analyze_owner_holdings", f"ANALYZE {OWNER_HOLDINGS_TABLE};")
build.run("analyze_company", f"ANALYZE {COMPANY_TABLE};")
build.run("analyze_names", f"ANALYZE {NAME_TABLE};")

✅ **[owner_build] idx_company_name**: `ok` | `CREATE INDEX` | 0.2s

✅ **[owner_build] idx_company_sic**: `ok` | `CREATE INDEX` | 0.0s

✅ **[owner_build] idx_name_cik**: `ok` | `CREATE INDEX` | 0.0s

✅ **[owner_build] analyze_owner_holdings**: `ok` | `ANALYZE` | 0.0s

✅ **[owner_build] analyze_company**: `ok` | `ANALYZE` | 0.2s

✅ **[owner_build] analyze_names**: `ok` | `ANALYZE` | 0.0s

""


## 5. Checks

The checks join the company table with `input_1.edgar_submissions` (`s`, EDGAR) and `input_1.owner_holdings`
(`h`, holdings coverage). Owners = companies with a 13F filing or holdings.

In [16]:
chk = p.stage("owner_checks")

chk.run("companies_by_group", f"""
    SELECT is_13f_filler,
           is_owner,
           sic IS NOT NULL                                          AS with_sic,
           count(*)                                                   AS companies,
           count(*) FILTER (WHERE cardinality(tickers) > 0)         AS with_ticker,
           count(*) FILTER (WHERE name IS NULL)                     AS name_null,
           count(*) FILTER (WHERE ein  IS NULL)                     AS ein_null,
           count(*) FILTER (WHERE lei  IS NOT NULL)                 AS lei_present
    FROM {COMPANY_TABLE}
    GROUP BY 1, 2, 3
    ORDER BY 1 DESC, 2 DESC, 3 DESC;
""", note="Previous test: 13F filers 20,108 (10,661 owners) | other companies 67,901 (66,841 with SIC, 1,060 ticker only)")

chk.run("is_owner_check", f"""
    -- is_owner must agree with the holdings coverage table
    SELECT count(*) FILTER (WHERE o.is_owner)                          AS owners,
           count(*) FILTER (WHERE o.is_owner <> (h.cik IS NOT NULL))   AS mismatches
    FROM {COMPANY_TABLE} o
    LEFT JOIN {OWNER_HOLDINGS_TABLE} h USING (cik);
""", note="Previous test: owners 10,661 | mismatches 0")

✅ **[owner_checks] companies_by_group**: `ok` | `SELECT 6` | rows: **6** | 0.0s

> Previous test: 13F filers 20,108 (10,661 owners) | other companies 67,901 (66,841 with SIC, 1,060 ticker only)

,is_13f_filler,is_owner,with_sic,companies,with_ticker,name_null,ein_null,lei_present
0,True,True,True,363,159,0,60,0
1,True,True,False,10318,10,0,2496,26
2,True,False,True,202,85,0,55,0
3,True,False,False,9225,1,0,3368,198
4,False,False,True,66841,6845,1,25114,19
5,False,False,False,1060,1060,0,787,2


✅ **[owner_checks] is_owner_check**: `ok` | `SELECT 1` | rows: **1** | 0.0s

> Previous test: owners 10,661 | mismatches 0

,owners,mismatches
0,10681,0


,owners,mismatches
0,10681,0


### Coverage of the holdings

Every holdings CIK gets a row by construction (full join in 4.2). What matters is whether EDGAR knows it:
`holdings_owner_not_in_edgar` should be 0, otherwise those owners have no name. `filings_without_owner`: CIKs of
`clean_up_1.filings` missing from the owner table (must be 0). `filings_without_holdings`: owners whose filings
were all removed by the holdings cleaning.

In [17]:
chk.run("holdings_coverage", f"""
    SELECT (SELECT count(*) FROM {OWNER_HOLDINGS_TABLE})                                AS owners_in_holdings,
           (SELECT count(DISTINCT f.cik) FROM {FILINGS_TABLE} f
             WHERE NOT EXISTS (SELECT 1 FROM {COMPANY_TABLE} o WHERE o.cik = f.cik))      AS filings_without_owner,
           (SELECT count(DISTINCT f.cik) FROM {FILINGS_TABLE} f
             WHERE NOT EXISTS (SELECT 1 FROM {OWNER_HOLDINGS_TABLE} h WHERE h.cik = f.cik)) AS filings_without_holdings,
           (SELECT count(*) FROM {OWNER_HOLDINGS_TABLE} h
             WHERE NOT EXISTS (SELECT 1 FROM {SUBMISSIONS_TABLE} s WHERE s.cik = h.cik)) AS holdings_owner_not_in_edgar;
""", note="Previous test: 10,661 owners in holdings | filings_without_owner 0 | filings_without_holdings 52 | "
          "holdings_owner_not_in_edgar 0")

✅ **[owner_checks] holdings_coverage**: `ok` | `SELECT 1` | rows: **1** | 0.3s

> Previous test: 10,661 owners in holdings | filings_without_owner 0 | filings_without_holdings 52 | holdings_owner_not_in_edgar 0

,owners_in_holdings,filings_without_owner,filings_without_holdings,holdings_owner_not_in_edgar
0,10681,0,29,0


,owners_in_holdings,filings_without_owner,filings_without_holdings,holdings_owner_not_in_edgar
0,10681,0,29,0


### Renamed owners and shared names

`name` is today's EDGAR name. `renamed_after_first_holding`: owners with a former name that was valid during
or after their first holdings quarter and differs from the current one, so the report-time name differs
from `name`. Duplicate names: one name used by several owners (e.g. a parent and its advisers).

In [18]:
chk.run("former_names", f"""
    SELECT count(*)                                                            AS former_name_rows,
           count(DISTINCT f.cik)                                               AS companies_with_former_names,
           count(DISTINCT f.cik) FILTER (WHERE f.name <> o.name)               AS companies_renamed,
           count(DISTINCT f.cik) FILTER (WHERE f.name <> o.name
                                           AND f.name_to >= h.first_rdate)     AS renamed_after_first_holding
    FROM {NAME_TABLE} f
    JOIN {COMPANY_TABLE} o USING (cik)
    LEFT JOIN {OWNER_HOLDINGS_TABLE} h USING (cik)
    WHERE NOT f.is_current;
""", note="Previous test: 30,240 rows | 20,948 companies | renamed 20,778 | renamed_after_first_holding 2,136")

chk.run("duplicate_owner_names", f"""
    SELECT o.name, count(*) AS ciks, string_agg(o.cik::text, ', ' ORDER BY o.cik) AS cik_list,
           count(h.cik) AS in_holdings
    FROM {COMPANY_TABLE} o
    JOIN {SUBMISSIONS_TABLE} s USING (cik)
    LEFT JOIN {OWNER_HOLDINGS_TABLE} h USING (cik)
    WHERE o.name IS NOT NULL
      AND (s.n_13f_hr + s.n_13f_hr_a + s.n_13f_nt + s.n_13f_nt_a > 0 OR h.cik IS NOT NULL)
    GROUP BY o.name
    HAVING count(*) > 1
    ORDER BY ciks DESC, o.name
    LIMIT 50;
""")

✅ **[owner_checks] former_names**: `ok` | `SELECT 1` | rows: **1** | 0.0s

> Previous test: 30,240 rows | 20,948 companies | renamed 20,778 | renamed_after_first_holding 2,136

,former_name_rows,companies_with_former_names,companies_renamed,renamed_after_first_holding
0,30240,20948,20778,2138


✅ **[owner_checks] duplicate_owner_names**: `ok` | `SELECT 31` | rows: **31** | 0.1s

,name,ciks,cik_list,in_holdings
0,"Aft, Forsyth & Sober, LLC",2,"1566801, 1569656",2
1,ARTEMIS INVESTMENT MANAGEMENT LLC,2,"838826, 1157287",2
2,"Bison Wealth, LLC",2,"1633172, 1845251",1
3,"Braddock Partners, LP",2,"1406947, 1434996",0
4,CFW CAPITAL MANAGEMENT INC,2,"706338, 1135160",2
5,CREDIT SUISSE SECURITIES (USA) LLC,2,"318336, 1230485",0
6,DWS Investment GmbH,2,"1167714, 1167717",0
7,FIRST NATIONAL TRUST CO,2,"1140769, 1277303",2
8,FRONTIER CAPITAL MANAGEMENT CO LLC,2,"351173, 1216488",2
9,GAVEA INVESTIMENTOS LTDA,2,"1454519, 1523705",2


,name,ciks,cik_list,in_holdings
0,"Aft, Forsyth & Sober, LLC",2,"1566801, 1569656",2
1,ARTEMIS INVESTMENT MANAGEMENT LLC,2,"838826, 1157287",2
2,"Bison Wealth, LLC",2,"1633172, 1845251",1
3,"Braddock Partners, LP",2,"1406947, 1434996",0
4,CFW CAPITAL MANAGEMENT INC,2,"706338, 1135160",2
5,CREDIT SUISSE SECURITIES (USA) LLC,2,"318336, 1230485",0
6,DWS Investment GmbH,2,"1167714, 1167717",0
7,FIRST NATIONAL TRUST CO,2,"1140769, 1277303",2
8,FRONTIER CAPITAL MANAGEMENT CO LLC,2,"351173, 1216488",2
9,GAVEA INVESTIMENTOS LTDA,2,"1454519, 1523705",2


In [19]:
chk.row_estimate(COMPANY_TABLE, "company")
chk.table_size(COMPANY_TABLE, "company")
chk.table_size(NAME_TABLE, "company_name")
chk.table_size(OWNER_HOLDINGS_TABLE, "owner_holdings")
chk.table_size(SUBMISSIONS_TABLE, "submissions")

✅ **[owner_checks] company_count_exact**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,n
0,88009


✅ **[owner_checks] company_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,19 MB,12 MB,6824 kB


✅ **[owner_checks] company_name_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

...

✅ **[owner_checks] owner_holdings_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,832 kB,552 kB,256 kB


✅ **[owner_checks] submissions_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,25 MB,23 MB,1944 kB


,total,heap,indexes
0,25 MB,23 MB,1944 kB


## 6. Run log analytics

Reads `steps.jsonl` from **all** runs in `logs/owner/`.

In [20]:
logs = load_logs(LOG_ROOT)
steps_table(logs, p.run_id)

,seq,phase,step,status,statusmessage,rowcount,expected,diff,duration_s
118,1,connection,connection_check_server,ok,None,NaN,NaN,NaN,0.054
119,2,connection,connection_check_schemas,ok,None,NaN,NaN,NaN,NaN
120,3,connection,connection_check_tables,ok,None,NaN,NaN,NaN,NaN
121,4,owner_create,drop_submissions_table,ok,DROP TABLE,-1.0,NaN,NaN,0.020
122,5,owner_create,create_submissions_table,ok,CREATE TABLE,-1.0,NaN,NaN,0.017
123,6,owner_load,holdings_ciks,ok,SELECT 10710,10710.0,10713.0,-3.0,0.291
124,7,owner_load,preflight_submissions,ok,None,NaN,NaN,NaN,NaN
125,8,owner_load,load_submissions,ok,None,88009.0,NaN,NaN,75.293
126,9,owner_profile,submissions_count_exact,ok,SELECT 1,1.0,NaN,NaN,0.013
127,10,owner_profile,submissions_totals,ok,SELECT 1,1.0,NaN,NaN,0.020


In [21]:
# Row counts that differ from the previous run
expected_mismatches(logs, p.run_id)

,seq,phase,step,status,statusmessage,rowcount,expected,diff,duration_s
123,6,owner_load,holdings_ciks,ok,SELECT 10710,10710.0,10713.0,-3.0,0.291
132,15,owner_build,insert_owner_holdings,ok,INSERT 0 10681,10681.0,10661.0,20.0,100.930


In [22]:
phase_durations(logs, p.run_id)

,seconds
phase,
connection,0.054
owner_create,0.037
owner_load,75.584
owner_profile,0.198
owner_build,102.625
owner_checks,0.510


In [23]:
compare_runs(logs)

rowcount                                                      duration_s                                                
run_id                                 20261009_171912 20261011_145639 20261011_151358 20261029_155949 20261009_171912 20261011_145639 20261011_151358 20261029_155949
phase         step                                                                                                                                                    
connection    connection_check_server              NaN             NaN             NaN             NaN           0.013           0.014           0.012           0.054
owner_create  drop_submissions_table              -1.0            -1.0            -1.0            -1.0           0.000           0.000           0.006           0.020
              create_submissions_table            -1.0            -1.0            -1.0            -1.0           0.013           0.011           0.008           0.017
owner_load    holdings_ciks                    10713.0         10713.0         10713.0         10710.0           0.031           0.027           0.027           0.291
              load_submissions                 20108.0         88009.0         88009.0         88009.0          45.083         104.445          94.793          75.293
owner_profile submissions_count_exact              1.0             1.0             1.0             1.0           0.002           0.004           0.148           0.013
              submissions_totals                   1.0             1.0             1.0             1.0           0.004           0.076           0.018           0.020
              entity_types                         3.0             3.0             3.0             3.0           0.005           0.017           0.021           0.039
              identifier_quality                   1.0             1.0             1.0             1.0           0.024           0.053           0.077           0.080
              first_13f_per_year                  37.0            37.0            37.0            37.0           0.005           0.017           0.039           0.046
owner_build   create_owner_holdings               -1.0            -1.0            -1.0            -1.0           0.010           0.009           0.013           0.009
              insert_owner_holdings            10661.0         10661.0         10661.0         10681.0         116.183          97.821         105.627         100.930
              create_owner_table                  -1.0             NaN             NaN             NaN           0.039             NaN             NaN             NaN
              insert_owner                     20108.0             NaN             NaN             NaN           0.058             NaN             NaN             NaN
              create_former_name_table            -1.0             NaN             NaN             NaN           0.039             NaN             NaN             NaN
              insert_former_names               4640.0             NaN             NaN             NaN           0.043             NaN             NaN             NaN
              idx_owner_name                      -1.0             NaN             NaN             NaN           0.030             NaN             NaN             NaN
              idx_former_name_cik                 -1.0             NaN             NaN             NaN           0.008             NaN             NaN             NaN
              analyze_owner_holdings              -1.0            -1.0            -1.0            -1.0           0.006           0.006           0.026           0.030
              analyze_owner                       -1.0             NaN             NaN             NaN           0.028             NaN             NaN             NaN
              analyze_former_name                 -1.0             NaN             NaN             NaN           0.005             NaN             NaN             NaN
              create_company_table    

In [24]:
p.close()

Run log: <THESIS_DIR>/src/Clean_solution/Database/Initialisation/logs/owner/20261029_155949/steps.jsonl
